# 02 — Enforce ODCS contract and build Silver

**Before running:** attach `lh_nyc_taxi_contracts` as the default Lakehouse, run Notebook 01, and upload `contracts/nyc_taxi_silver.odcs.yaml` to `Files/contracts/`.

The notebook reads the ODCS contract to verify declared columns/required fields, standardizes names and types, routes policy violations to quarantine, writes Silver, and records contract check outcomes. It raises an exception if a blocking check fails.


In [ ]:
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import yaml

CONTRACT_PATH = Path("/lakehouse/default/Files/contracts/nyc_taxi_silver.odcs.yaml")
if not CONTRACT_PATH.exists():
    raise FileNotFoundError(
        f"ODCS contract not found at {CONTRACT_PATH}. Upload contracts/nyc_taxi_silver.odcs.yaml to Lakehouse Files/contracts/ first."
    )
with open(CONTRACT_PATH, "r", encoding="utf-8") as handle:
    contract = yaml.safe_load(handle)

if contract.get("apiVersion") != "v3.1.0" or contract.get("kind") != "DataContract":
    raise ValueError("Expected an ODCS v3.1.0 DataContract file.")

contract_objects = contract.get("schema", [])
if not contract_objects:
    raise ValueError("The contract must declare at least one schema object.")
contract_object = next(
    (obj for obj in contract_objects if obj.get("name") == "silver_yellow_taxi_trips"),
    None,
)
if contract_object is None:
    raise ValueError("Contract must declare schema object silver_yellow_taxi_trips.")
properties = contract_object.get("properties", [])
contract_columns = [p["name"] for p in properties if p.get("name")]
required_columns = [p["name"] for p in properties if p.get("required") is True and p.get("name")]
print(f"Loaded contract {contract['name']} v{contract['version']}")
print("Contract columns:", len(contract_columns), "Required fields:", len(required_columns))

bronze = spark.table("bronze_yellow_taxi_trips")
# Source column names have varied in capitalization over time; match case-insensitively.
source_names = {col_name.lower(): col_name for col_name in bronze.columns}
rename_candidates = {
    "vendorid": "vendor_id",
    "tpep_pickup_datetime": "pickup_datetime",
    "tpep_dropoff_datetime": "dropoff_datetime",
    "passenger_count": "passenger_count",
    "trip_distance": "trip_distance",
    "ratecodeid": "rate_code_id",
    "store_and_fwd_flag": "store_and_fwd_flag",
    "pulocationid": "pickup_location_id",
    "dolocationid": "dropoff_location_id",
    "payment_type": "payment_type",
    "fare_amount": "fare_amount",
    "extra": "extra",
    "mta_tax": "mta_tax",
    "tip_amount": "tip_amount",
    "tolls_amount": "tolls_amount",
    "improvement_surcharge": "improvement_surcharge",
    "total_amount": "total_amount",
    "congestion_surcharge": "congestion_surcharge",
    "airport_fee": "airport_fee",
    "_source_file": "source_file",
    "_ingested_at": "ingested_at",
}
for source_key, canonical_name in rename_candidates.items():
    actual_name = source_names.get(source_key)
    if actual_name and actual_name != canonical_name:
        bronze = bronze.withColumnRenamed(actual_name, canonical_name)

# These source attributes are needed for the governed trip product.
source_required = [
    "pickup_datetime", "dropoff_datetime", "trip_distance", "pickup_location_id",
    "dropoff_location_id", "fare_amount", "total_amount", "source_file", "ingested_at"
]
missing_source = sorted(set(source_required) - set(bronze.columns))
if missing_source:
    raise ValueError(
        "Source schema is missing expected fields: " + ", ".join(missing_source)
        + ". TLC source schemas can change; inspect the Bronze columns and update the mapping/contract deliberately."
    )

# Cast canonical types. Optional charge columns are created as nulls if absent.
integer_columns = ["vendor_id", "passenger_count", "rate_code_id", "pickup_location_id", "dropoff_location_id", "payment_type"]
amount_columns = ["trip_distance", "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount", "congestion_surcharge", "airport_fee"]
for name in integer_columns:
    if name in bronze.columns:
        bronze = bronze.withColumn(name, F.col(name).cast("int"))
for name in amount_columns:
    if name in bronze.columns:
        bronze = bronze.withColumn(name, F.col(name).cast("double"))
    else:
        bronze = bronze.withColumn(name, F.lit(None).cast("double"))
for name in ["pickup_datetime", "dropoff_datetime", "ingested_at"]:
    bronze = bronze.withColumn(name, F.col(name).cast("timestamp"))
if "store_and_fwd_flag" not in bronze.columns:
    bronze = bronze.withColumn("store_and_fwd_flag", F.lit(None).cast("string"))
else:
    bronze = bronze.withColumn("store_and_fwd_flag", F.col("store_and_fwd_flag").cast("string"))

bronze = (bronze
    .withColumn("trip_date", F.to_date("pickup_datetime"))
    .withColumn(
        "trip_duration_minutes",
        (F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long")) / F.lit(60.0)
    ))

# Build a human-readable list of rejection reasons. Null payment_type is permitted;
# only a present value outside the source-defined code domain is rejected.
required_row_fields = [
    "pickup_datetime", "dropoff_datetime", "pickup_location_id", "dropoff_location_id",
    "trip_distance", "fare_amount", "total_amount"
]
missing_required_condition = None
for name in required_row_fields:
    condition = F.col(name).isNull()
    missing_required_condition = condition if missing_required_condition is None else (missing_required_condition | condition)

reason_parts = [
    F.when(missing_required_condition, F.lit("MISSING_REQUIRED_FIELDS")),
    F.when(
        F.col("pickup_datetime").isNotNull() & F.col("dropoff_datetime").isNotNull()
        & (F.col("dropoff_datetime") <= F.col("pickup_datetime")),
        F.lit("INVALID_TRIP_TIME_ORDER")
    ),
    F.when(
        F.col("trip_distance").isNotNull()
        & ((F.col("trip_distance") <= 0) | (F.col("trip_distance") > 100)),
        F.lit("TRIP_DISTANCE_OUT_OF_POLICY_RANGE")
    ),
    F.when(
        (F.col("fare_amount") < 0) | (F.col("total_amount") < 0),
        F.lit("NEGATIVE_FARE_OR_TOTAL_AMOUNT")
    ),
    F.when(
        F.col("payment_type").isNotNull() & (~F.col("payment_type").isin(0, 1, 2, 3, 4, 5, 6)),
        F.lit("PAYMENT_TYPE_OUTSIDE_DOMAIN")
    ),
]
with_reasons = bronze.withColumn("quality_reason", F.concat_ws(";", *reason_parts))
quarantine = with_reasons.filter(F.length(F.col("quality_reason")) > 0)
valid = with_reasons.filter(F.length(F.col("quality_reason")) == 0)

# Only publish fields declared by the contract; fail rather than silently ignore contract drift.
available_canonical = set(valid.columns)
missing_contract_columns = sorted(set(contract_columns) - available_canonical)
if missing_contract_columns:
    raise ValueError("Contract columns are not produced by the notebook: " + ", ".join(missing_contract_columns))

silver = valid.select(*contract_columns).persist(StorageLevel.MEMORY_AND_DISK)
quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("quarantine_yellow_taxi_trips")
# Validate the Silver candidate before publishing it to the consumer-facing table.
# Derive contract checks in one aggregate pass over the Silver candidate.
metrics = silver.agg(
    F.count(F.lit(1)).alias("row_count"),
    F.sum(F.when(F.col("pickup_datetime").isNull(), 1).otherwise(0)).alias("null_pickup_datetime"),
    F.sum(F.when(F.col("dropoff_datetime").isNull(), 1).otherwise(0)).alias("null_dropoff_datetime"),
    F.sum(F.when(F.col("pickup_datetime") >= F.col("dropoff_datetime"), 1).otherwise(0)).alias("invalid_time_order"),
    F.sum(F.when((F.col("trip_distance") <= 0) | (F.col("trip_distance") > 100) | F.col("trip_distance").isNull(), 1).otherwise(0)).alias("invalid_distance"),
    F.sum(F.when((F.col("fare_amount") < 0) | (F.col("total_amount") < 0) | F.col("fare_amount").isNull() | F.col("total_amount").isNull(), 1).otherwise(0)).alias("invalid_amounts"),
    F.sum(F.when(F.col("payment_type").isNotNull() & (~F.col("payment_type").isin(0, 1, 2, 3, 4, 5, 6)), 1).otherwise(0)).alias("invalid_payment_type"),
).first().asDict()
metrics = {key: (int(value) if value is not None else 0) for key, value in metrics.items()}
quarantine_reason_missing = spark.table("quarantine_yellow_taxi_trips").filter(F.col("quality_reason").isNull() | (F.length(F.col("quality_reason")) == 0)).limit(1).count()
missing_contract = sorted(set(contract_columns) - set(silver.columns))
missing_required = sorted(set(required_columns) - set(silver.columns))

checks = [
    ("contract_columns_present", 0 if not missing_contract else len(missing_contract), "must equal 0", not missing_contract),
    ("required_contract_fields_declared_in_output", 0 if not missing_required else len(missing_required), "must equal 0", not missing_required),
    ("silver_row_count_nonzero", metrics["row_count"], "must be greater than 0", metrics["row_count"] > 0),
    ("pickup_datetime_not_null", metrics["null_pickup_datetime"], "must equal 0", metrics["null_pickup_datetime"] == 0),
    ("dropoff_datetime_not_null", metrics["null_dropoff_datetime"], "must equal 0", metrics["null_dropoff_datetime"] == 0),
    ("pickup_before_dropoff", metrics["invalid_time_order"], "must equal 0", metrics["invalid_time_order"] == 0),
    ("trip_distance_policy", metrics["invalid_distance"], "must equal 0", metrics["invalid_distance"] == 0),
    ("nonnegative_fare_and_total", metrics["invalid_amounts"], "must equal 0", metrics["invalid_amounts"] == 0),
    ("payment_type_domain", metrics["invalid_payment_type"], "must equal 0", metrics["invalid_payment_type"] == 0),
    ("quarantine_reason_populated", quarantine_reason_missing, "must equal 0", quarantine_reason_missing == 0),
]
from datetime import datetime, timezone
run_timestamp = datetime.now(timezone.utc).replace(tzinfo=None)
results = [
    (run_timestamp, contract.get("id"), contract.get("version"), name, int(observed), expectation, bool(passed))
    for name, observed, expectation, passed in checks
]
result_df = spark.createDataFrame(
    results,
    ["evaluated_at_utc", "contract_id", "contract_version", "check_name", "observed_value", "expectation", "passed"]
)
result_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dq_contract_results")

summary = {
    "source_rows": spark.table("bronze_yellow_taxi_trips").count(),
    "candidate_silver_rows": metrics["row_count"],
    "quarantined_rows": spark.table("quarantine_yellow_taxi_trips").count(),
    "blocking_checks_passed": sum(1 for row in checks if row[3]),
    "blocking_checks_total": len(checks),
}
print(summary)
display(spark.table("dq_contract_results").orderBy(F.col("passed").asc(), "check_name"))
display(spark.table("quarantine_yellow_taxi_trips").select("quality_reason").groupBy("quality_reason").count().orderBy(F.desc("count")))

failed_checks = [row[0] for row in checks if not row[3]]
if failed_checks:
    silver.unpersist()
    raise ValueError("Contract validation failed; Silver was not overwritten and downstream publication must stop. Failed checks: " + ", ".join(failed_checks))

# Publish only after every blocking check passes.
silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_yellow_taxi_trips")
silver.unpersist()
print("Contract validation passed; Silver was published and is ready for Gold publication.")
